# ANÁLISIS COMPLETO WAGE11
## Limpieza, Preprocesamiento y Modelado Predictivo de Datos de Salarios

---

### Descripción del Proyecto

Este notebook realiza un **análisis integral y documentado** de los datos de salarios contenidos en `Wage11.xlsx`.

**Etapas del análisis:**
1. Exploración inicial de datos
2. Limpieza - Eliminación de duplicados
3. Análisis y visualización de valores faltantes
4. Imputación de valores faltantes
5. Normalización y estandarización (con visualizaciones)
6. Ingeniería de características
7. Análisis Exploratorio (EDA) - Heatmaps, Scatter plots, Grupos
8. Preparación y división train/test (80/20)
9. Modelado predictivo - Regresión Lineal Múltiple
10. Evaluación del modelo (R², MAE, RMSE, overfitting, residuos)
11. Conclusiones y recomendaciones
12. Guardar archivo procesado

**Dataset:** Wage11.xlsx | **Variables:** 15 | **Observaciones:** 935

## SECCIÓN 1: IMPORTAR LIBRERÍAS

Importación de todas las librerías necesarias para el análisis.

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd               # Manipulación de DataFrames
import numpy as np                # Operaciones numéricas
import matplotlib.pyplot as plt   # Visualizaciones base
import seaborn as sns             # Gráficas estadísticas

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# Configuración visual global
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

print('=' * 70)
print('  LIBRERÍAS IMPORTADAS CORRECTAMENTE')
print('=' * 70)

## SECCIÓN 2: CARGAR DATOS

Carga del archivo Excel y primera inspección del dataset.

In [ ]:
# ============================================================
# CARGAR ARCHIVO EXCEL
# ============================================================

file_path = 'Wage11.xlsx'
df = pd.read_excel(file_path)
df_original = df.copy()   # copia de respaldo sin modificar

print('=' * 70)
print('  ARCHIVO CARGADO EXITOSAMENTE')
print('=' * 70)
print(f'\n  Dimensiones  : {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'  Variables    : {list(df.columns)}')

## SECCIÓN 3: EXPLORACIÓN INICIAL

Análisis de estructura, tipos de datos y estadísticas descriptivas.

In [ ]:
# Primeras filas
print('PRIMERAS 5 FILAS:')
print(df.head().to_string())

In [ ]:
# Tipos de datos e información general
print('TIPOS DE DATOS:')
print(df.dtypes)
print(f'\nTotal registros : {len(df)}')
print(f'Total columnas  : {len(df.columns)}')

In [ ]:
# Estadísticas descriptivas
print('ESTADÍSTICAS DESCRIPTIVAS:')
print(df.describe().round(2).to_string())

In [ ]:
# ============================================================
# DISTRIBUCIÓN DE LA VARIABLE OBJETIVO: WAGE
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma con media y mediana
axes[0].hist(df['wage'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución de WAGE (Salario)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Salario ($)')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df['wage'].mean(), color='red', linestyle='--',
                label=f'Media: {df["wage"].mean():.0f}')
axes[0].axvline(df['wage'].median(), color='orange', linestyle='--',
                label=f'Mediana: {df["wage"].median():.0f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['wage'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Boxplot de WAGE', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Salario ($)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DISTRIBUCIONES DE VARIABLES NUMÉRICAS CLAVE
# ============================================================
vars_hist = ['educ', 'exper', 'IQ', 'KWW', 'age', 'hours']
labels = ['Educación (años)', 'Experiencia (años)', 'Cociente Intelectual',
          'KWW Score', 'Edad', 'Horas Semanales']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, var, lbl in zip(axes.flat, vars_hist, labels):
    ax.hist(df[var].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(df[var].mean(), color='red', linestyle='--', linewidth=1.2,
               label=f'Media: {df[var].mean():.1f}')
    ax.set_title(f'Distribución de {lbl}', fontweight='bold')
    ax.set_xlabel(lbl)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.suptitle('Distribuciones de Variables Numéricas Clave', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## SECCIÓN 4: DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS

**Justificación:** Los registros duplicados distorsionan el análisis estadístico, aumentan artificialmente el peso de ciertas observaciones y generan sesgos en el modelo. Se eliminan conservando el primer registro de cada grupo.

In [ ]:
# ============================================================
# DETECTAR DUPLICADOS (excluimos 'id' que es identificador único)
# ============================================================
duplicados_mask = df.duplicated(subset=df.columns.difference(['id']), keep=False)
num_dup = df.duplicated(subset=df.columns.difference(['id'])).sum()

print('=' * 70)
print('  ANÁLISIS DE DUPLICADOS')
print('=' * 70)
print(f'\n  Duplicados encontrados : {num_dup}')
print(f'  Porcentaje             : {num_dup/len(df)*100:.2f}%')

if num_dup > 0:
    print(f'\n  Muestra de duplicados:')
    print(df[duplicados_mask].head(3).to_string())

In [ ]:
# Eliminar duplicados y resetear índice
df = df.drop_duplicates(
    subset=df.columns.difference(['id']),
    keep='first'
).reset_index(drop=True)

print('RESULTADO DESPUÉS DE ELIMINAR DUPLICADOS:')
print(f'  Registros originales          : {len(df_original)}')
print(f'  Registros después de limpieza : {len(df)}')
print(f'  Registros eliminados          : {len(df_original) - len(df)}')
print(f'  Nuevas dimensiones            : {df.shape[0]} x {df.shape[1]}')

## SECCIÓN 5: ANÁLISIS DE VALORES FALTANTES

Identificación de variables con datos incompletos y sus porcentajes.

In [ ]:
# ============================================================
# TABLA DE VALORES FALTANTES
# ============================================================
faltantes = pd.DataFrame({
    'Variable'   : df.columns,
    'Faltantes'  : df.isnull().sum().values,
    'Porcentaje' : (df.isnull().sum().values / len(df) * 100).round(2)
})
faltantes_solo = faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

print('=' * 60)
print('  ANÁLISIS DE VALORES FALTANTES')
print('=' * 60)
if len(faltantes_solo) > 0:
    print('\nVariables con valores faltantes:')
    print(faltantes_solo.to_string(index=False))
else:
    print('\n  No hay valores faltantes en el dataset')
print(f'\n  Total valores faltantes: {df.isnull().sum().sum()}')

In [ ]:
# ============================================================
# VISUALIZACIÓN DE VALORES FALTANTES
# ============================================================
faltantes_plot = faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=True)

if len(faltantes_plot) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(faltantes_plot['Variable'], faltantes_plot['Porcentaje'],
                   color='salmon', edgecolor='white', alpha=0.85)
    ax.set_title('Porcentaje de Valores Faltantes por Variable',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Porcentaje (%)')
    for bar, pct in zip(bars, faltantes_plot['Porcentaje']):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'{pct}%', va='center', fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print('No hay valores faltantes que graficar - dataset completo')

## SECCIÓN 6: IMPUTACIÓN DE VALORES FALTANTES

**Estrategia:**
- **MEDIANA** para `wage`, `educ`, `exper`, `black`, `sibs` → robusta ante outliers y distribuciones asimétricas
- **MEDIA** para `IQ`, `KWW` → distribuciones aproximadamente simétricas

In [ ]:
# ============================================================
# IMPUTACIÓN CON MEDIANA
# ============================================================
vars_mediana = ['wage', 'educ', 'exper', 'black', 'sibs']

print('Variables imputadas con MEDIANA (robustas ante outliers):')
for var in vars_mediana:
    n_missing = df[var].isnull().sum()
    if n_missing > 0:
        mediana = df[var].median()
        df[var].fillna(mediana, inplace=True)
        print(f'  {var:<12} → mediana = {mediana:.2f}  ({n_missing} valores imputados)')
    else:
        print(f'  {var:<12} → sin faltantes')

# ============================================================
# IMPUTACIÓN CON MEDIA
# ============================================================
vars_media = ['IQ', 'KWW']

print('\nVariables imputadas con MEDIA (distribuciones simétricas):')
for var in vars_media:
    n_missing = df[var].isnull().sum()
    if n_missing > 0:
        media = df[var].mean()
        df[var].fillna(media, inplace=True)
        print(f'  {var:<12} → media = {media:.2f}  ({n_missing} valores imputados)')
    else:
        print(f'  {var:<12} → sin faltantes')

print(f'\nFaltantes totales después de imputación: {df.isnull().sum().sum()}')

## SECCIÓN 7: NORMALIZACIÓN Y ESTANDARIZACIÓN

- **MinMaxScaler [0, 1]** aplicado a `WAGE`
- **StandardScaler (z-score)** aplicado a `KWW`

In [ ]:
# ============================================================
# NORMALIZACIÓN MINMAXSCALER - WAGE
# ============================================================
scaler_wage = MinMaxScaler()
df['wage_normalizado'] = scaler_wage.fit_transform(df[['wage']])

print('NORMALIZACIÓN DE WAGE (MinMaxScaler [0,1]):')
print(f'  Original   - min: {df["wage"].min():.2f}  '
      f'max: {df["wage"].max():.2f}  media: {df["wage"].mean():.2f}')
print(f'  Norm [0,1] - min: {df["wage_normalizado"].min():.4f}  '
      f'max: {df["wage_normalizado"].max():.4f}  media: {df["wage_normalizado"].mean():.4f}')

# ============================================================
# ESTANDARIZACIÓN STANDARDSCALER - KWW
# ============================================================
scaler_kww = StandardScaler()
df['KWW_estandarizado'] = scaler_kww.fit_transform(df[['KWW']])

print('\nESTANDARIZACIÓN DE KWW (StandardScaler z-score):')
print(f'  Original  - media: {df["KWW"].mean():.2f}  std: {df["KWW"].std():.2f}')
print(f'  Estándar  - media: {df["KWW_estandarizado"].mean():.6f}  '
      f'std: {df["KWW_estandarizado"].std():.4f}')

In [ ]:
# ============================================================
# VISUALIZACIÓN ANTES vs DESPUÉS
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# WAGE original
axes[0,0].hist(df['wage'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,0].set_title('WAGE - Original', fontweight='bold')
axes[0,0].set_xlabel('Salario ($)')

# WAGE normalizado
axes[0,1].hist(df['wage_normalizado'], bins=40, color='seagreen', edgecolor='white', alpha=0.8)
axes[0,1].set_title('WAGE - Normalizado [0,1]', fontweight='bold')
axes[0,1].set_xlabel('Valor normalizado')

# KWW original
axes[1,0].hist(df['KWW'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1,0].set_title('KWW - Original', fontweight='bold')
axes[1,0].set_xlabel('KWW')

# KWW estandarizado
axes[1,1].hist(df['KWW_estandarizado'], bins=30, color='seagreen', edgecolor='white', alpha=0.8)
axes[1,1].set_title('KWW - Estandarizado (z-score)', fontweight='bold')
axes[1,1].set_xlabel('Z-score')

plt.suptitle('Transformaciones: Antes vs Después', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## SECCIÓN 8: INGENIERÍA DE CARACTERÍSTICAS

Creación de variables derivadas que capturan relaciones importantes:
- **salary_per_hour**: productividad salarial por hora trabajada
- **educ_exper_ratio**: balance entre educación formal y experiencia laboral

In [ ]:
# ============================================================
# FEATURE 1: SALARIO POR HORA
# ============================================================
df['salary_per_hour'] = df['wage'] / df['hours'].replace(0, np.nan)

print('FEATURE 1: salary_per_hour = wage / hours')
print(f'  Promedio : {df["salary_per_hour"].mean():.4f}')
print(f'  Mediana  : {df["salary_per_hour"].median():.4f}')

# ============================================================
# FEATURE 2: RATIO EDUCACIÓN / EXPERIENCIA
# ============================================================
df['educ_exper_ratio'] = df['educ'] / (df['exper'] + 1)

print('\nFEATURE 2: educ_exper_ratio = educ / (exper + 1)')
print(f'  Promedio : {df["educ_exper_ratio"].mean():.4f}')
print(f'  Mediana  : {df["educ_exper_ratio"].median():.4f}')

In [ ]:
# Visualización de las nuevas features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['salary_per_hour'].dropna(), bins=40, color='mediumpurple',
             edgecolor='white', alpha=0.8)
axes[0].set_title('Distribución: Salario por Hora', fontweight='bold')
axes[0].set_xlabel('salary_per_hour')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(df['educ_exper_ratio'].dropna(), bins=40, color='mediumpurple',
             edgecolor='white', alpha=0.8)
axes[1].set_title('Distribución: Ratio Educación/Experiencia', fontweight='bold')
axes[1].set_xlabel('educ_exper_ratio')
axes[1].set_ylabel('Frecuencia')

plt.suptitle('Nuevas Variables Creadas (Feature Engineering)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## SECCIÓN 9: ANÁLISIS EXPLORATORIO DE DATOS (EDA)

Visualizaciones extensas: correlaciones, scatter plots, análisis por grupos.

In [ ]:
# ============================================================
# MATRIZ DE CORRELACIONES - HEATMAP
# ============================================================
vars_num = ['wage', 'educ', 'exper', 'IQ', 'KWW', 'age', 'hours',
            'tenure', 'salary_per_hour', 'educ_exper_ratio']

corr_matrix = df[vars_num].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Matriz de Correlaciones entre Variables Numéricas',
          fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CORRELACIONES CON WAGE (GRÁFICA DE BARRAS)
# ============================================================
vars_interes = ['educ', 'exper', 'IQ', 'KWW', 'age', 'hours', 'tenure',
                'salary_per_hour', 'educ_exper_ratio', 'married', 'black', 'south', 'urban']

corrs = df[vars_interes + ['wage']].corr()['wage'][vars_interes].sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['salmon' if c < 0 else 'steelblue' for c in corrs]
corrs.plot(kind='barh', ax=ax, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Correlación de cada Variable con WAGE', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
for i, (val, label) in enumerate(zip(corrs, corrs.index)):
    ax.text(val + (0.01 if val >= 0 else -0.01), i, f'{val:.3f}',
            va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SCATTER PLOTS: WAGE vs VARIABLES CLAVE
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
vars_scatter = [
    ('educ',   'Educación (años)'),
    ('exper',  'Experiencia (años)'),
    ('IQ',     'Cociente Intelectual'),
    ('age',    'Edad'),
    ('tenure', 'Antigüedad'),
    ('hours',  'Horas Semanales')
]

for ax, (var, label) in zip(axes.flat, vars_scatter):
    valid = df[[var, 'wage']].dropna()
    ax.scatter(valid[var], valid['wage'], alpha=0.3, color='steelblue', s=15)
    m, b = np.polyfit(valid[var], valid['wage'], 1)
    x_line = np.linspace(valid[var].min(), valid[var].max(), 100)
    ax.plot(x_line, m * x_line + b, color='red', linewidth=1.5,
            label=f'β = {m:.2f}')
    ax.set_title(f'WAGE vs {label}', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Salario ($)')
    ax.legend(fontsize=9)

plt.suptitle('Relación entre WAGE y Variables Predictoras', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ANÁLISIS POR GRUPOS DEMOGRÁFICOS
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Por estado civil
married_stats = df.groupby('married')['wage'].mean()
axes[0].bar(['No Casado', 'Casado'], married_stats.values,
            color=['salmon', 'steelblue'], alpha=0.85, edgecolor='white')
axes[0].set_title('Salario Promedio por Estado Civil', fontweight='bold')
axes[0].set_ylabel('Salario promedio ($)')
for i, v in enumerate(married_stats.values):
    axes[0].text(i, v + 20, f'${v:.0f}', ha='center', fontweight='bold')

# Por zona
urban_stats = df.groupby('urban')['wage'].mean()
axes[1].bar(['Rural', 'Urbano'], urban_stats.values,
            color=['salmon', 'steelblue'], alpha=0.85, edgecolor='white')
axes[1].set_title('Salario Promedio: Rural vs Urbano', fontweight='bold')
axes[1].set_ylabel('Salario promedio ($)')
for i, v in enumerate(urban_stats.values):
    axes[1].text(i, v + 20, f'${v:.0f}', ha='center', fontweight='bold')

# Por región
south_stats = df.groupby('south')['wage'].mean()
axes[2].bar(['Norte/Otros', 'Sur'], south_stats.values,
            color=['salmon', 'steelblue'], alpha=0.85, edgecolor='white')
axes[2].set_title('Salario Promedio: Norte vs Sur', fontweight='bold')
axes[2].set_ylabel('Salario promedio ($)')
for i, v in enumerate(south_stats.values):
    axes[2].text(i, v + 20, f'${v:.0f}', ha='center', fontweight='bold')

plt.suptitle('Análisis Comparativo por Grupos Demográficos',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# BOXPLOTS: WAGE POR NIVEL EDUCATIVO
# ============================================================
educ_groups = []
educ_labels = []
for educ_val in sorted(df['educ'].dropna().unique()):
    vals = df[df['educ'] == educ_val]['wage'].dropna()
    if len(vals) >= 5:
        educ_groups.append(vals.values)
        educ_labels.append(str(int(educ_val)))

plt.figure(figsize=(14, 5))
plt.boxplot(educ_groups, labels=educ_labels, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
plt.title('Distribución de Salarios por Años de Educación',
          fontsize=13, fontweight='bold')
plt.xlabel('Años de Educación')
plt.ylabel('Salario ($)')
plt.tight_layout()
plt.show()

## SECCIÓN 10: PREPARACIÓN Y DIVISIÓN DE DATOS

Selección de variables predictoras y división train/test (80%/20%).

In [ ]:
# ============================================================
# SELECCIONAR VARIABLES
# ============================================================
X = df[['educ', 'exper', 'hours', 'IQ', 'KWW', 'tenure', 'age',
        'married', 'black', 'south', 'urban',
        'salary_per_hour', 'educ_exper_ratio']].copy()
y = df['wage'].copy()

# Eliminar filas con NaN residuales
idx = X.notna().all(axis=1) & y.notna()
X = X[idx].reset_index(drop=True)
y = y[idx].reset_index(drop=True)

print(f'Variables predictoras (X): {len(X.columns)}')
print(f'  {list(X.columns)}')
print(f'\nVariable objetivo (y): wage')
print(f'Total muestras: {len(X)}')

# División 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'\nDivisión Train/Test:')
print(f'  Entrenamiento (80%): {len(X_train)} muestras  |  Salario prom: ${y_train.mean():.2f}')
print(f'  Prueba        (20%): {len(X_test)} muestras  |  Salario prom: ${y_test.mean():.2f}')

## SECCIÓN 11: ENTRENAMIENTO DEL MODELO

Regresión Lineal Múltiple con 13 variables predictoras.

In [ ]:
# ============================================================
# ENTRENAR MODELO DE REGRESIÓN LINEAL
# ============================================================
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print('Modelo entrenado exitosamente')
print(f'Intercepto (β₀): {modelo.intercept_:.4f}')

# Tabla de coeficientes
coefs = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': modelo.coef_
}).sort_values('Coeficiente', ascending=False)

print('\nCOEFICIENTES DEL MODELO:')
print(f'{"Variable":<25} {"Coeficiente":>15}')
print('-' * 42)
for _, row in coefs.iterrows():
    signo = '+' if row['Coeficiente'] >= 0 else ''
    print(f'  {row["Variable"]:<23} {signo}{row["Coeficiente"]:>13.4f}')

In [ ]:
# ============================================================
# VISUALIZAR COEFICIENTES DEL MODELO
# ============================================================
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['steelblue' if c > 0 else 'salmon' for c in coefs['Coeficiente']]
ax.barh(coefs['Variable'], coefs['Coeficiente'],
        color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coeficientes del Modelo de Regresión Lineal',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Valor del coeficiente')
plt.tight_layout()
plt.show()

## SECCIÓN 12: EVALUACIÓN DEL MODELO

Métricas de desempeño en entrenamiento y prueba. Análisis de overfitting y residuos.

In [ ]:
# ============================================================
# CALCULAR MÉTRICAS
# ============================================================
y_train_pred = modelo.predict(X_train)
y_test_pred  = modelo.predict(X_test)

r2_train   = r2_score(y_train, y_train_pred)
r2_test    = r2_score(y_test, y_test_pred)
mae_train  = mean_absolute_error(y_train, y_train_pred)
mae_test   = mean_absolute_error(y_test, y_test_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

print('=' * 65)
print(f'  {"MÉTRICA":<35} {"ENTRENAMIENTO":>12}  {"PRUEBA":>10}')
print('=' * 65)
print(f'  {"R² (Coef. Determinación)":<35} {r2_train:>12.4f}  {r2_test:>10.4f}')
print(f'  {"MAE (Error Absoluto Medio)":<35} ${mae_train:>11.2f}  ${mae_test:>9.2f}')
print(f'  {"RMSE (Raíz Error Cuadrático)":<35} ${rmse_train:>11.2f}  ${rmse_test:>9.2f}')
print('=' * 65)

diff = r2_train - r2_test
print(f'\n  Diferencia R² (overfitting): {diff:.4f}')
if diff < 0.05:
    print('  → EXCELENTE: Sin overfitting significativo')
elif diff < 0.10:
    print('  → BUENO: Overfitting mínimo')
else:
    print('  → ADVERTENCIA: Posible overfitting')

In [ ]:
# ============================================================
# GRÁFICA: REAL vs PREDICHO
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Test set
axes[0].scatter(y_test, y_test_pred, alpha=0.4, color='steelblue', s=25)
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--',
             linewidth=1.5, label='Predicción perfecta')
axes[0].set_title(f'Real vs Predicho (Prueba) - R²={r2_test:.4f}', fontweight='bold')
axes[0].set_xlabel('Valor Real ($)')
axes[0].set_ylabel('Valor Predicho ($)')
axes[0].legend()

# Train set
axes[1].scatter(y_train, y_train_pred, alpha=0.3, color='seagreen', s=20)
axes[1].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
             'r--', linewidth=1.5, label='Predicción perfecta')
axes[1].set_title(f'Real vs Predicho (Entrenamiento) - R²={r2_train:.4f}', fontweight='bold')
axes[1].set_xlabel('Valor Real ($)')
axes[1].set_ylabel('Valor Predicho ($)')
axes[1].legend()

plt.suptitle('Comparación: Valores Reales vs Predichos', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# ANÁLISIS DE RESIDUOS
# ============================================================
residuos = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuos vs predichos
axes[0].scatter(y_test_pred, residuos, alpha=0.4, color='steelblue', s=25)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuos vs Valores Predichos', fontweight='bold')
axes[0].set_xlabel('Valores Predichos ($)')
axes[0].set_ylabel('Residuos ($)')

# Distribución de residuos
axes[1].hist(residuos, bins=35, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].axvline(residuos.mean(), color='orange', linestyle='--', linewidth=1.5,
                label=f'Media: {residuos.mean():.2f}')
axes[1].set_title('Distribución de Residuos', fontweight='bold')
axes[1].set_xlabel('Residuo ($)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Análisis de Residuos del Modelo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Media de residuos : {residuos.mean():.4f} (debe ser ~0)')
print(f'Std de residuos   : {residuos.std():.4f}')

## SECCIÓN 13: CONCLUSIONES Y RECOMENDACIONES

In [ ]:
print("=" * 70)
print("  CONCLUSIONES FINALES DEL ANÁLISIS")
print("=" * 70)

resumen = f"""

  1. LIMPIEZA Y PREPARACIÓN
     - Registros originales            : {len(df_original)}
     - Registros después de limpieza   : {len(df)}
     - Duplicados eliminados           : {len(df_original) - len(df)}

  2. TRANSFORMACIONES REALIZADAS
     - WAGE normalizado                : MinMaxScaler [0, 1]
     - KWW estandarizado               : StandardScaler (media=0, std=1)
     - Features creadas                : salary_per_hour, educ_exper_ratio

  3. MODELO PREDICTIVO
     - Tipo                            : Regresión Lineal Múltiple
     - Variables predictoras           : {len(X.columns)}
     - Muestras de entrenamiento       : {len(X_train)}
     - Muestras de prueba              : {len(X_test)}

  4. RENDIMIENTO
     - R² (Prueba)                     : {r2_test:.4f} ({r2_test*100:.2f}%)
     - MAE (Prueba)                    : ${mae_test:.2f} por empleado
     - RMSE (Prueba)                   : ${rmse_test:.2f}
     - Overfitting (R² train - test)   : {r2_train - r2_test:.4f}

  5. EVALUACIÓN
     - {"SIN overfitting - modelo CONFIABLE" if (r2_train - r2_test) < 0.05 else "Overfitting mínimo - modelo aceptable"}
"""
print(resumen)

print("  TOP 5 VARIABLES MÁS INFLUYENTES EN EL SALARIO:")
for i, (_, row) in enumerate(coefs.head(5).iterrows(), 1):
    print(f"     {i}. {row['Variable']:<25} β = {row['Coeficiente']:+.4f}")

print()
print("  RECOMENDACIONES:")
print("     1. Usar el modelo para estimación salarial de nuevos empleados")
print("     2. Actualizar el modelo periódicamente con nuevos datos")
print("     3. Explorar modelos no lineales (Random Forest, XGBoost) para mejorar R²")
print("=" * 70)

## SECCIÓN 14: GUARDAR ARCHIVO PROCESADO

Guardar el dataset limpio y transformado en formato Excel.

In [ ]:
# ============================================================
# GUARDAR EN EXCEL
# ============================================================
output_file = 'Wage11_PROCESADO.xlsx'
df.to_excel(output_file, index=False)

print('=' * 60)
print('  ARCHIVO PROCESADO GUARDADO')
print('=' * 60)
print(f'\n  Archivo   : {output_file}')
print(f'  Registros : {len(df)}')
print(f'  Columnas  : {len(df.columns)}')
print(f'\n  Columnas incluidas:')
for i, col in enumerate(df.columns, 1):
    print(f'    {i:2d}. {col}')
print()
print('  ANÁLISIS COMPLETADO EXITOSAMENTE')